## Compare Hugging Face open-weight models locally with Ollama

The original preflight uses three public Hugging Face checkpoints downloaded to your computer and run locally with Transformers: Qwen3 0.6B, Qwen2.5 1.5B Instruct, and SmolLM2 1.7B Instruct. It compares them with your Ollama `llama3.1:8b` across all 16 directed guesser/answerer pairs, including separate-session self-pairs.

**No Hugging Face token or hosted inference is used, so this notebook makes no paid HF API calls.** The model weights are downloaded from Hugging Face on first use and cached locally; this uses disk, download bandwidth, and your PC's compute/electricity. The original 48-game preflight (one repeat ? 16 pairs ? 3 current targets) validates the pipeline, not the final study; its results are saved/resumed in `free_hf_ollama_games.jsonl`. The final cell runs a separate, same-size comparison using Phi-3.5 Mini, IBM Granite 3.3 2B, and Qwen2.5 3B alongside Ollama, saved in `additional_hf_ollama_games.jsonl`.

In [18]:
import os

# Remove a token from the earlier hosted-inference setup; this notebook now runs locally only.
os.environ.pop("HF_TOKEN", None)
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
globals().pop("token", None)

os.environ.setdefault("OLLAMA_BASE_URL", "http://localhost:11434/v1")
os.environ.setdefault("OLLAMA_MODEL", "llama3.1:8b")
os.environ.setdefault("OLLAMA_API_KEY", "ollama")

%pip -q install openai transformers accelerate
import subprocess
import sys

cuda_check = subprocess.run(
    [sys.executable, "-c", "import torch; print(int(torch.cuda.is_available()))"],
    capture_output=True,
    text=True,
)
if cuda_check.returncode != 0 or cuda_check.stdout.strip() != "1":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps",
         "torch==2.10.0", "--index-url", "https://download.pytorch.org/whl/cu128"],
        check=True,
    )
    raise RuntimeError("Installed CUDA-enabled PyTorch. Restart the notebook kernel, then rerun this setup cell.")

print("No Hugging Face token or hosted inference is needed.")
print("CUDA-capable runner:", cuda_check.stdout.strip() == "1")
if cuda_check.stdout.strip() == "1":
    gpu = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.get_device_name(0))"],
                         capture_output=True, text=True, check=True)
    print("GPU:", gpu.stdout.strip())
print(f"Ollama model: {os.environ['OLLAMA_MODEL']} at {os.environ['OLLAMA_BASE_URL']}")


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
No Hugging Face token or hosted inference is needed.
CUDA-capable runner: True
GPU: NVIDIA GeForce RTX 5060 Ti
Ollama model: llama3.1:8b at http://localhost:11434/v1


In [19]:
from pathlib import Path

script_path = Path.cwd() / "twenty_questions_pilot.py"
if not script_path.is_file():
    raise FileNotFoundError(f"Runner not found: {script_path}")

print(f"Runner ready: {script_path}")
print("HF checkpoints: Qwen3-0.6B, Qwen2.5-1.5B-Instruct, SmolLM2-1.7B-Instruct (local inference)")
print("Ollama model:", __import__("os").getenv("OLLAMA_MODEL", "llama3.1:8b"))

Runner ready: c:\Users\vinay\OneDrive\Desktop\DSF Group Project\twenty_questions_pilot.py
HF checkpoints: Qwen3-0.6B, Qwen2.5-1.5B-Instruct, SmolLM2-1.7B-Instruct (local inference)
Ollama model: llama3.1:8b


In [22]:
from pathlib import Path
import subprocess
import sys
import json
from collections import Counter

script_path = Path.cwd() / "twenty_questions_pilot.py"
print("Starting 48-game local preflight: 4 models ? 4 directed pairs ? 3 targets.")
print("Hugging Face weights may download on first use; inference stays on this computer and makes no hosted API calls.")
subprocess.run(
    [sys.executable, str(script_path), "--repeats", "1", "--results-file", "free_hf_ollama_games.jsonl"],
    cwd=script_path.parent,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

results_path = script_path.parent / "free_hf_ollama_games.jsonl"
games = [json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines() if line.strip()]
pairings = {(g["answerer"], g["guesser"]) for g in games}
targets = Counter(g["secret"] for g in games)
if len(games) != 48 or len({g["game_id"] for g in games}) != 48 or len(pairings) != 16:
    raise RuntimeError("Preflight results are incomplete or contain duplicate game IDs.")
if sorted(targets.values()) != [16, 16, 16]:
    raise RuntimeError(f"Unexpected target balance: {dict(targets)}")

wins = sum(g["won"] for g in games)
answer_count = sum(g["answer_turns"] for g in games)
invalid_count = sum(g["invalid_answers"] for g in games)
models = sorted({g["answerer"] for g in games} | {g["guesser"] for g in games})
print(f"Validated {len(games)} unique games across {len(pairings)} directed pairs; targets: {dict(targets)}")
print(f"Overall wins: {wins}/{len(games)} ({wins / len(games):.1%}); invalid answers: {invalid_count}/{answer_count} ({invalid_count / answer_count:.1%})")
print("Win rate by guesser:")
for model in models:
    group = [g for g in games if g["guesser"] == model]
    print(f"  {model}: {sum(g['won'] for g in group)}/{len(group)} ({sum(g['won'] for g in group) / len(group):.1%})")
print("Invalid-answer rate by answerer:")
for model in models:
    group = [g for g in games if g["answerer"] == model]
    answers = sum(g["answer_turns"] for g in group)
    invalid = sum(g["invalid_answers"] for g in group)
    print(f"  {model}: {invalid}/{answers} ({invalid / answers:.1%})")

Starting 48-game local preflight: 4 models ? 4 directed pairs ? 3 targets.
Hugging Face weights may download on first use; inference stays on this computer and makes no hosted API calls.
Validated 48 unique games across 16 directed pairs; targets: {'banana': 16, 'bicycle': 16, 'elephant': 16}
Overall wins: 2/48 (4.2%); invalid answers: 91/729 (12.5%)
Win rate by guesser:
  Llama (Ollama): 1/12 (8.3%)
  Qwen2.5 1.5B (HF local): 1/12 (8.3%)
  Qwen3 0.6B (HF local): 0/12 (0.0%)
  SmolLM2 1.7B (HF local): 0/12 (0.0%)
Invalid-answer rate by answerer:
  Llama (Ollama): 13/190 (6.8%)
  Qwen2.5 1.5B (HF local): 78/188 (41.5%)
  Qwen3 0.6B (HF local): 0/187 (0.0%)
  SmolLM2 1.7B (HF local): 0/164 (0.0%)


In [24]:
from pathlib import Path
import subprocess
import sys
import json
from collections import Counter

script_path = Path.cwd() / "twenty_questions_pilot.py"
results_name = "additional_hf_ollama_games.jsonl"
print("Starting 48-game additional-model preflight: 3 Hugging Face models + Ollama; all 16 directed pairs and 3 targets.")
print("All Hugging Face models run locally; no hosted inference API or HF token is used.")
run = subprocess.run(
    [sys.executable, str(script_path), "--model-set", "additional", "--repeats", "1",
     "--results-file", results_name],
    cwd=script_path.parent,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)
print(run.stdout)

results_path = script_path.parent / results_name
games = [json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines() if line.strip()]
pairings = {(g["answerer"], g["guesser"]) for g in games}
targets = Counter(g["secret"] for g in games)
if len(games) != 48 or len({g["game_id"] for g in games}) != 48 or len(pairings) != 16:
    raise RuntimeError("Additional-model results are incomplete or contain duplicate game IDs.")
if sorted(targets.values()) != [16, 16, 16]:
    raise RuntimeError(f"Unexpected target balance: {dict(targets)}")

wins = sum(g["won"] for g in games)
answer_count = sum(g["answer_turns"] for g in games)
invalid_count = sum(g["invalid_answers"] for g in games)
models = sorted({g["answerer"] for g in games} | {g["guesser"] for g in games})
print(f"Validated {len(games)} unique games across {len(pairings)} pairs; targets: {dict(targets)}")
print(f"Overall wins: {wins}/{len(games)} ({wins / len(games):.1%}); invalid answers: {invalid_count}/{answer_count} ({invalid_count / answer_count:.1%})")
print("Win rate by guesser:")
for model in models:
    group = [g for g in games if g["guesser"] == model]
    print(f"  {model}: {sum(g['won'] for g in group)}/{len(group)} ({sum(g['won'] for g in group) / len(group):.1%})")
print("Invalid-answer rate by answerer:")
for model in models:
    group = [g for g in games if g["answerer"] == model]
    answers = sum(g["answer_turns"] for g in group)
    invalid = sum(g["invalid_answers"] for g in group)
    print(f"  {model}: {invalid}/{answers} ({invalid / answers:.1%})")

Starting 48-game additional-model preflight: 3 Hugging Face models + Ollama; all 16 directed pairs and 3 targets.
All Hugging Face models run locally; no hosted inference API or HF token is used.
All Hugging Face checkpoints run locally; no hosted inference API or HF token is used.
48 games already saved; playing 0 more...

===== Paste these into the proposal =====
[N_GAMES] = 48
[WIN_OVERALL] = 21%
[AVG_Q] = 11.4
[INVALID_RATE] = 8.5%

Bad guesser moves (empty or broken questions) per guesser:
  Phi-3.5 Mini (HF local): 0
  Granite 3.3 2B (HF local): 0
  Qwen2.5 3B (HF local): 0
  Llama (Ollama): 2

Win-rate table (rows = answerer, columns = guesser):
| Answerer / Guesser | Phi-3.5 Mini (HF local) | Granite 3.3 2B (HF local) | Qwen2.5 3B (HF local) | Llama (Ollama) |
| Phi-3.5 Mini (HF local) | 1/3 | 0/3 | 0/3 | 2/3 |
| Granite 3.3 2B (HF local) | 1/3 | 0/3 | 0/3 | 1/3 |
| Qwen2.5 3B (HF local) | 0/3 | 0/3 | 0/3 | 1/3 |
| Llama (Ollama) | 2/3 | 1/3 | 1/3 | 0/3 |

Full game logs saved 